In [ ]:
# Sample grid cells, perform threshold analysis, and calculate mean threshold
# This script samples n grid cells, performs graphical analysis for each,
# and calculates the mean threshold to use for the entire region

import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from pyextremes import EVA
import pickle
from pathlib import Path

# Configuration
n_sample_cells = 20  # Number of grid cells to sample
percentile_range = range(85, 99)  # Range of percentiles to test
output_dir = Path('/Users/jschaper/Documents/Codes/gesdisc-tutorials/conference-presentations/agu-2026/Data/threshold_analysis')
output_dir.mkdir(exist_ok=True)

# Read data
ds = xr.open_dataset("/Users/jschaper/Documents/Codes/gesdisc-tutorials/conference-presentations/agu-2026/IMERG_subsets/subset_kerchunk.nc", engine="netcdf4")

# Randomly sample grid cell indices
np.random.seed(42)  # For reproducibility
total_lat = len(ds.lat)
total_lon = len(ds.lon)

# Generate random sample indices
sample_lat_indices = np.random.randint(0, total_lat, n_sample_cells)
sample_lon_indices = np.random.randint(0, total_lon, n_sample_cells)

print(f"Sampling {n_sample_cells} grid cells from {total_lat}×{total_lon} grid")
print(f"Sample indices (lat, lon): {list(zip(sample_lat_indices, sample_lon_indices))}")

# Store results for each sampled cell
sample_results = []

# Process each sampled grid cell
for idx, (i, j) in enumerate(zip(sample_lat_indices, sample_lon_indices)):
    print(f"\nProcessing sample {idx+1}/{n_sample_cells}: lat_idx={i}, lon_idx={j}")
    
    # Extract data for specific lat/lon
    df = ds.isel(lat=i, lon=j)['precipitation'].to_dataframe().reset_index()[['time', 'lat', 'lon', 'precipitation']]
    df.columns = ['date', 'lat', 'lon', 'precip_mm']
    
    lat_val = df['lat'].iloc[0]
    lon_val = df['lon'].iloc[0]
    
    # Filter out zero precipitation
    df_noz = df[df['precip_mm'] > 0].copy()
    df_noz = df_noz.set_index('date')['precip_mm']
    
    print(f"  Location: Lat={lat_val:.2f}, Lon={lon_val:.2f}")
    print(f"  Non-zero precipitation events: {len(df_noz)}")
    
    # Test different thresholds
    threshold_data = []
    
    for pct in percentile_range:
        thres = np.percentile(df_noz, pct)
        
        try:
            model = EVA(data=df_noz)
            model.get_extremes("POT", threshold=thres, r='48hr')
            
            n_extremes = len(model.extremes)
            
            # Skip if too few extremes
            if n_extremes < 10:
                continue
            
            model.fit_model(distribution="genpareto")
            params = model.model.fit_parameters
            
            threshold_data.append({
                'percentile': pct,
                'threshold_mm': thres,
                'n_extremes': n_extremes,
                'shape': params.get('c', np.nan),
                'scale': params.get('scale', np.nan),
                'loc': params.get('loc', np.nan)
            })
        except Exception as e:
            print(f"  Warning: Failed to fit model at {pct}th percentile: {e}")
            continue
    
    # Create diagnostic plots for this cell
    if len(threshold_data) > 0:
        df_thresh = pd.DataFrame(threshold_data)
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle(f'Threshold Analysis - Sample {idx+1}\nLat: {lat_val:.2f}°, Lon: {lon_val:.2f}° (Grid: {i},{j})', 
                     fontsize=14, fontweight='bold')
        
        # Plot 1: Threshold vs Percentile
        ax1 = axes[0, 0]
        ax1.plot(df_thresh['percentile'], df_thresh['threshold_mm'], 'o-', linewidth=2, markersize=6)
        ax1.set_xlabel('Percentile', fontsize=11)
        ax1.set_ylabel('Threshold (mm)', fontsize=11)
        ax1.set_title('Threshold Value vs Percentile', fontsize=12)
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Number of extremes
        ax2 = axes[0, 1]
        ax2.plot(df_thresh['threshold_mm'], df_thresh['n_extremes'], 'o-', linewidth=2, markersize=6, color='green')
        ax2.set_xlabel('Threshold (mm)', fontsize=11)
        ax2.set_ylabel('Number of Extreme Events', fontsize=11)
        ax2.set_title('Sample Size vs Threshold', fontsize=12)
        ax2.axhline(y=50, color='r', linestyle='--', alpha=0.5, label='Min recommended (50)')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Plot 3: Shape parameter
        ax3 = axes[1, 0]
        ax3.plot(df_thresh['threshold_mm'], df_thresh['shape'], 'o-', linewidth=2, markersize=6, color='purple')
        ax3.axhline(y=0, color='k', linestyle='-', alpha=0.3)
        ax3.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='ξ = ±0.5')
        ax3.axhline(y=-0.5, color='r', linestyle='--', alpha=0.5)
        ax3.set_xlabel('Threshold (mm)', fontsize=11)
        ax3.set_ylabel('Shape Parameter (ξ)', fontsize=11)
        ax3.set_title('GPD Shape Parameter', fontsize=12)
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # Plot 4: Scale parameter
        ax4 = axes[1, 1]
        ax4.plot(df_thresh['threshold_mm'], df_thresh['scale'], 'o-', linewidth=2, markersize=6, color='orange')
        ax4.set_xlabel('Threshold (mm)', fontsize=11)
        ax4.set_ylabel('Scale Parameter (σ)', fontsize=11)
        ax4.set_title('GPD Scale Parameter', fontsize=12)
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        # Save figure
        fig_path = output_dir / f'threshold_analysis_sample{idx+1}_lat{i}_lon{j}.png'
        plt.savefig(fig_path, dpi=300, bbox_inches='tight')
        print(f"  Saved diagnostic plot: {fig_path}")
        plt.close()
        
        # Select threshold for this cell (using 95th percentile as your original approach)
        selected_threshold = np.percentile(df_noz, 95)
        selected_row = df_thresh[df_thresh['percentile'] == 95]
        
        if len(selected_row) == 0:
            # If 95th percentile wasn't in our test range, find closest
            selected_row = df_thresh.iloc[(df_thresh['percentile'] - 95).abs().argsort()[:1]]
        
        sample_results.append({
            'sample_id': idx + 1,
            'lat_index': i,
            'lon_index': j,
            'latitude': lat_val,
            'longitude': lon_val,
            'threshold_mm': selected_threshold,
            'n_extremes': selected_row['n_extremes'].values[0] if len(selected_row) > 0 else np.nan,
            'shape': selected_row['shape'].values[0] if len(selected_row) > 0 else np.nan,
            'scale': selected_row['scale'].values[0] if len(selected_row) > 0 else np.nan,
            'n_nonzero': len(df_noz),
            'threshold_data': df_thresh  # Store full threshold analysis
        })
        
        print(f"  Selected threshold (95th percentile): {selected_threshold:.2f} mm")
        print(f"  Number of extremes at threshold: {selected_row['n_extremes'].values[0] if len(selected_row) > 0 else 'N/A'}")

# Create summary DataFrame
summary_df = pd.DataFrame([{k: v for k, v in result.items() if k != 'threshold_data'} 
                           for result in sample_results])

# Calculate statistics
mean_threshold = summary_df['threshold_mm'].mean()
std_threshold = summary_df['threshold_mm'].std()
cv_threshold = (std_threshold / mean_threshold) * 100  # Coefficient of variation

print("\n" + "="*70)
print("THRESHOLD ANALYSIS SUMMARY")
print("="*70)
print(f"\nNumber of sampled cells: {len(sample_results)}")
print(f"\nThreshold Statistics (95th percentile):")
print(f"  Mean threshold: {mean_threshold:.2f} mm")
print(f"  Std deviation: {std_threshold:.2f} mm")
print(f"  Coefficient of variation: {cv_threshold:.1f}%")
print(f"  Min threshold: {summary_df['threshold_mm'].min():.2f} mm")
print(f"  Max threshold: {summary_df['threshold_mm'].max():.2f} mm")
print(f"  Median threshold: {summary_df['threshold_mm'].median():.2f} mm")

print(f"\nShape Parameter Statistics:")
print(f"  Mean: {summary_df['shape'].mean():.3f}")
print(f"  Std: {summary_df['shape'].std():.3f}")
print(f"  Range: [{summary_df['shape'].min():.3f}, {summary_df['shape'].max():.3f}]")

print(f"\nNumber of Extremes Statistics:")
print(f"  Mean: {summary_df['n_extremes'].mean():.0f}")
print(f"  Min: {summary_df['n_extremes'].min():.0f}")
print(f"  Max: {summary_df['n_extremes'].max():.0f}")

# Assess variability
if cv_threshold > 30:
    print(f"\n⚠️  WARNING: High variability detected (CV = {cv_threshold:.1f}%)")
    print("   Consider using spatially-varying thresholds or climate-zone-based thresholds")
elif cv_threshold > 20:
    print(f"\n⚠️  CAUTION: Moderate variability detected (CV = {cv_threshold:.1f}%)")
    print("   Review diagnostic plots to ensure mean threshold is appropriate")
else:
    print(f"\n✓  Variability is acceptable (CV = {cv_threshold:.1f}%)")
    print("   Mean threshold appears suitable for regional application")

# Create summary visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Regional Threshold Analysis Summary (n={len(sample_results)} cells)', 
             fontsize=14, fontweight='bold')

# Plot 1: Histogram of thresholds
ax1 = axes[0, 0]
ax1.hist(summary_df['threshold_mm'], bins=15, edgecolor='black', alpha=0.7)
ax1.axvline(mean_threshold, color='r', linestyle='--', linewidth=2, label=f'Mean: {mean_threshold:.2f} mm')
ax1.axvline(summary_df['threshold_mm'].median(), color='g', linestyle='--', linewidth=2, 
            label=f'Median: {summary_df["threshold_mm"].median():.2f} mm')
ax1.set_xlabel('Threshold (mm)', fontsize=11)
ax1.set_ylabel('Frequency', fontsize=11)
ax1.set_title('Distribution of Thresholds', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Plot 2: Spatial distribution of thresholds
ax2 = axes[0, 1]
scatter = ax2.scatter(summary_df['longitude'], summary_df['latitude'], 
                     c=summary_df['threshold_mm'], s=100, cmap='viridis', 
                     edgecolors='black', linewidth=0.5)
ax2.set_xlabel('Longitude', fontsize=11)
ax2.set_ylabel('Latitude', fontsize=11)
ax2.set_title('Spatial Distribution of Thresholds', fontsize=12)
cbar = plt.colorbar(scatter, ax=ax2)
cbar.set_label('Threshold (mm)', fontsize=10)
ax2.grid(True, alpha=0.3)

# Plot 3: Threshold vs Number of extremes
ax3 = axes[1, 0]
ax3.scatter(summary_df['threshold_mm'], summary_df['n_extremes'], s=80, alpha=0.6, edgecolors='black')
ax3.set_xlabel('Threshold (mm)', fontsize=11)
ax3.set_ylabel('Number of Extreme Events', fontsize=11)
ax3.set_title('Threshold vs Sample Size', fontsize=12)
ax3.axhline(y=50, color='r', linestyle='--', alpha=0.5, label='Min recommended')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Shape parameter distribution
ax4 = axes[1, 1]
ax4.hist(summary_df['shape'], bins=15, edgecolor='black', alpha=0.7, color='purple')
ax4.axvline(summary_df['shape'].mean(), color='r', linestyle='--', linewidth=2, 
            label=f'Mean: {summary_df["shape"].mean():.3f}')
ax4.axvline(0, color='k', linestyle='-', alpha=0.3)
ax4.set_xlabel('Shape Parameter (ξ)', fontsize=11)
ax4.set_ylabel('Frequency', fontsize=11)
ax4.set_title('Distribution of Shape Parameters', fontsize=12)
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
summary_fig_path = output_dir / 'threshold_summary.png'
plt.savefig(summary_fig_path, dpi=300, bbox_inches='tight')
print(f"\nSaved summary plot: {summary_fig_path}")
plt.close()

# Save results
summary_excel_path = output_dir / 'threshold_analysis_summary.xlsx'
summary_df.to_excel(summary_excel_path, index=False, sheet_name='Sample Analysis')
print(f"Saved summary table: {summary_excel_path}")

# Save full results including threshold_data
pickle_path = output_dir / 'threshold_analysis_full.pkl'
with open(pickle_path, 'wb') as f:
    pickle.dump(sample_results, f)
print(f"Saved full results: {pickle_path}")

# Save the mean threshold to use in main analysis
threshold_config = {
    'mean_threshold': mean_threshold,
    'std_threshold': std_threshold,
    'cv_threshold': cv_threshold,
    'n_samples': len(sample_results),
    'percentile_used': 95,
    'sample_indices': list(zip(sample_lat_indices, sample_lon_indices))
}

config_path = output_dir / 'threshold_config.pkl'
with open(config_path, 'wb') as f:
    pickle.dump(threshold_config, f)
print(f"Saved threshold configuration: {config_path}")

print("\n" + "="*70)
print(f"RECOMMENDED THRESHOLD FOR REGIONAL ANALYSIS: {mean_threshold:.2f} mm")
print("="*70)